In [2]:
import pandas as pd

df = pd.read_excel("Online Retail.xlsx")
print(df.shape)
df.head()

(541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
print(df.isnull().sum())
print(df["CustomerID"].nunique())
print(df["StockCode"].nunique())

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64
4372
4070


In [4]:
print(df.isnull().sum())

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [5]:

df = df.dropna(subset=["CustomerID"])
print(df.shape)

(406829, 8)


In [6]:
invoicenoclean = df["InvoiceNo"].astype(str).str.startswith("C")
df = df[~invoicenoclean]
print(df.shape)

(397924, 8)


In [7]:
user_item = df.groupby(["CustomerID", "StockCode"])["Quantity"].sum().unstack(fill_value=0)
print(user_item.shape)

(4339, 3665)


In [8]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

item_matrix = user_item.T
item_similarity = cosine_similarity(item_matrix)
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item.columns,
    columns=user_item.columns
)

print(item_similarity_df.shape)

(3665, 3665)


In [9]:
def get_similar_items(stockcode, n=5):
    if stockcode not in item_similarity_df.columns:
        return "Product not found"
    
    similar = item_similarity_df[stockcode].sort_values(ascending=False)[1:n+1]
    
    descriptions = df[["StockCode", "Description"]].drop_duplicates(subset="StockCode").set_index("StockCode")
    
    results = pd.DataFrame({
        "StockCode": similar.index,
        "Similarity": similar.values
    })
    results["Description"] = results["StockCode"].map(descriptions["Description"])
    return results

print(get_similar_items("85123A"))

  StockCode  Similarity                       Description
0     21175    0.749651       GIN + TONIC DIET METAL SIGN
1     21733    0.658732  RED HANGING HEART T-LIGHT HOLDER
2     82552    0.643868               WASHROOM METAL SIGN
3     82551    0.642480            LAUNDRY 15C METAL SIGN
4     23288    0.630982         GREEN VINTAGE SPOT BEAKER


In [10]:
def recommend_for_customer(customer_id, n=5):
    if customer_id not in user_item.index:
        return "Customer not found"
    
    customer_purchases = user_item.loc[customer_id]
    bought = customer_purchases[customer_purchases > 0].index.tolist()
    
    scores = pd.Series(0.0, index=user_item.columns)
    for item in bought:
        scores += item_similarity_df[item]
    
    scores = scores.drop(bought)
    top_n = scores.sort_values(ascending=False).head(n)
    
    descriptions = df[["StockCode", "Description"]].drop_duplicates(subset="StockCode").set_index("StockCode")
    
    results = pd.DataFrame({
        "StockCode": top_n.index,
        "Score": top_n.values
    })
    results["Description"] = results["StockCode"].map(descriptions["Description"])
    return results

print(recommend_for_customer(12346.0))

  StockCode     Score                        Description
0     23165  0.008958      LARGE CERAMIC TOP STORAGE JAR
1     22652  0.007583                  TRAVEL SEWING KIT
2     22985  0.007535       WRAP, BILLBOARD FONTS DESIGN
3     21984  0.007294   PACK OF 12 PINK PAISLEY TISSUES 
4     21980  0.007018  PACK OF 12 RED RETROSPOT TISSUES 


In [11]:
print(user_item.index[:5])

Index([12346.0, 12347.0, 12348.0, 12349.0, 12350.0], dtype='float64', name='CustomerID')


In [18]:
print(df["Quantity"].describe())
print(df["Quantity"].quantile([0.95, 0.99]))

count    378293.000000
mean          7.460482
std           7.431880
min           1.000000
25%           2.000000
50%           4.000000
75%          12.000000
max          36.000000
Name: Quantity, dtype: float64
0.95    24.0
0.99    33.0
Name: Quantity, dtype: float64


In [14]:
q95 = df["Quantity"].quantile(0.95)
df = df[df["Quantity"] <= q95]
print(df.shape)
print(df["Quantity"].describe())

(378293, 8)
count    378293.000000
mean          7.460482
std           7.431880
min           1.000000
25%           2.000000
50%           4.000000
75%          12.000000
max          36.000000
Name: Quantity, dtype: float64


In [15]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import cross_validate

reader = Reader(rating_scale=(1, df["Quantity"].max()))

data = Dataset.load_from_df(
    df[["CustomerID", "StockCode", "Quantity"]],
    reader
)

model = SVD(n_factors=50, random_state=42)
results = cross_validate(model, data, measures=["RMSE", "MAE"], cv=3, verbose=True)

print(f"Mean RMSE: {results['test_rmse'].mean():.3f}")
print(f"Mean MAE: {results['test_mae'].mean():.3f}")

Evaluating RMSE, MAE of algorithm SVD on 3 split(s).

                  Fold 1  Fold 2  Fold 3  Mean    Std     
RMSE (testset)    4.3924  4.4070  4.3706  4.3900  0.0150  
MAE (testset)     2.6351  2.6398  2.6238  2.6329  0.0067  
Fit time          2.57    2.58    2.69    2.61    0.05    
Test time         0.85    0.87    0.87    0.86    0.01    
Mean RMSE: 4.390
Mean MAE: 2.633


In [16]:
from surprise import Dataset, Reader, SVD

reader = Reader(rating_scale=(1, df["Quantity"].max()))
data = Dataset.load_from_df(df[["CustomerID", "StockCode", "Quantity"]], reader)

trainset = data.build_full_trainset()
model.fit(trainset)

def recommend_svd(customer_id, n=5):
    all_items = df["StockCode"].unique()
    bought = df[df["CustomerID"] == customer_id]["StockCode"].unique()
    not_bought = [item for item in all_items if item not in bought]
    
    predictions = [(item, model.predict(customer_id, item).est) for item in not_bought]
    predictions.sort(key=lambda x: x[1], reverse=True)
    
    top_n = predictions[:n]
    descriptions = df[["StockCode", "Description"]].drop_duplicates(subset="StockCode").set_index("StockCode")
    
    results = pd.DataFrame(top_n, columns=["StockCode", "Predicted Quantity"])
    results["Description"] = results["StockCode"].map(descriptions["Description"])
    return results

customer = df["CustomerID"].iloc[0]
print(recommend_svd(customer))

   StockCode  Predicted Quantity                         Description
0      22492           18.797363             MINI PAINT SET VINTAGE 
1      22610           16.641180            PENS ASSORTED FUNNY FACE
2      23310           16.166623             BUBBLEGUM RING ASSORTED
3      22693           15.063832  GROW A FLYTRAP OR SUNFLOWER IN TIN
4      23077           14.767080                 DOUGHNUT LIP GLOSS 


In [17]:
customer = df["CustomerID"].iloc[0]

print("Item-Item Collaborative Filtering:")
print(recommend_for_customer(customer))
print("\nSVD Matrix Factorisation:")
print(recommend_svd(customer))

Item-Item Collaborative Filtering:
  StockCode     Score                          Description
0     23284  3.183389        DOORMAT KEEP CALM AND COME IN
1     22767  2.948651          TRIPLE PHOTO FRAME CORNICE 
2     21428  2.944595  SET3 BOOK BOX GREEN GINGHAM FLOWER 
3     22365  2.846777            DOORMAT RESPECTABLE HOUSE
4     22960  2.798959             JAM MAKING SET WITH JARS

SVD Matrix Factorisation:
   StockCode  Predicted Quantity                         Description
0      22492           18.797363             MINI PAINT SET VINTAGE 
1      22610           16.641180            PENS ASSORTED FUNNY FACE
2      23310           16.166623             BUBBLEGUM RING ASSORTED
3      22693           15.063832  GROW A FLYTRAP OR SUNFLOWER IN TIN
4      23077           14.767080                 DOUGHNUT LIP GLOSS 
